In [ ]:
#----------Broadcasting
import numpy as np

item_prices = np.array([[100, 200, 300],
                                [150, 250, 350]])
taxes = np.array([[0.1, 0.2, 0.3],[0.15,0.7,0.35]])
fixed_taxes = taxes.astype(float)
total_prices = item_prices * (1 + taxes)

print(total_prices)

[[110.  240.  390. ]
 [172.5 425.  472.5]]


In [ ]:
# ----------Vectorized distance calculation
vectors = np.array([
    [1, 2],
    [3, 4],
    [5, 6],
    [8, 9]
])

query = np.array([2, 3])
difference = vectors - query                # <<-- This is called broadcasting.
"""
[1, 2] - [2, 3] = [-1, -1]                  # NumPy automatically subtracts [2, 3] from every row: So the point is 1 unit left and 1 unit down from the query.
[3, 4] - [2, 3] = [ 1,  1]
[5, 6] - [2, 3] = [ 3,  3]
[8, 9] - [2, 3] = [ 6,  6]
"""
distances = np.linalg.norm(difference, axis = 1)       # - How far is each of these points from (2,3)? - For every point, calculate how far it is from the query point [2,3].
print(distances)
"""
Why x² + y²? -- Pythagorean theorem.
              (2,3) Query
                 ●
                /|
               / |
              /  | 1
             /   |
            ●---- 
          (1,2)
             1

distance = √(horizontal² + vertical²) = √(1² + 1²) = 1.41421356
"""
#  Pythagoras  → The document is X units away from the query.
#  Cosine similarity → Are they pointing in the same direction? and why it is common to use cosine similarity in embedding--
# Because with many embedding models, the direction of the vector is intended to represent semantic meaning, while the 
# magnitude is often less important.
"""
                Query
                  ↓
             Embedding
                  ↓
       ┌──────────┴──────────┐
       ↓                     ↓
Cosine similarity       Other signals
       ↓                     ↓
 semantic match       metadata / keyword /
                      freshness / filters (E.g. BM25)
       └──────────┬──────────┘
                  ↓
             Final ranking
                  ↓
             Top-K documents
"""

[1.41421356 1.41421356 4.24264069 8.48528137]


'\nWhy x² + y²? -- Pythagorean theorem.\n              (2,3) Query\n                 ●\n                /|\n               / |\n              /  | 1\n             /   |\n            ●---- \n          (1,2)\n             1\n\ndistance = √(horizontal² + vertical²)\n'

In [ ]:
#----------Cosine similarity - How similar is each document vector to the query vector?
""" 
             A · B                     # <<-- Dot product
cosine =  -----------
           ||A|| ||B||                 # << -- lengths of the vectors after they normalized
"""
import numpy as np
documents = np.array([        
    [1, 0, 1],
    [0, 1, 1],
    [1, 1, 0]
])
user_query = np.array([1, 0, 1])
"""
              dim 0   dim 1   dim 2
            ┌───────┬───────┬───────┐
Document 0  │   1   │   0   │   1   │
Document 1  │   0   │   1   │   1   │
Document 2  │   1   │   1   │   0   │
            └───────┴───────┴───────┘
shape = (3,3)

user_query -- [1, 0, 1]
shape = (3,)
"""
documents_norm = documents / np.linalg.norm(              # << -- finds the length (magnitude) of each vector, and then dividing by that length normalizes the vector.     
    documents,
    axis=1,                                               # <<-- Normalize each row  
    keepdims=True
)
""" after nomalization we get below - 
documents_norm =
[0.6, 0.8]
[1.0, 0.0]
"""
query_norm = user_query / np.linalg.norm(user_query)      # << -- lengths of the vectors / # Normalize each document row   
""" after normalization of query_norm becomes -
[0.6, 0.8]
"""
scores = documents_norm @ query_norm                      # << --- # is cosine similarity when the vectors are normalized first.
indices = np.argsort(scores)[-2:][::-1]    # <<---  [-2:] - Take top 2, [::-1] - Put highest score first

print("Scores:", scores)
print("Top documents:", indices)
print("Top scores:", scores[indices])

""" Conceptually -- 
Document 0 ─────────────→ Query
[1,0,1]                  [1,0,1]
     │
     └──── very similar

Document 1 ─────────────→ Query
[0,1,1]                  [1,0,1]
     │
     └──── partially similar

Document 2 ─────────────→ Query
[1,1,0]                  [1,0,1]
     │
     └──── partially similar

scores
[1.00, 0.50, 0.50]         # <<- So Document 0 has cosine similarity 1.0 because it is exactly the same direction as the query.
shape = (3,)
"""


Scores: [1.  0.5 0.5]
Top documents: [0 2]
Top scores: [1.  0.5]


' Conceptually -- \nDocument 0 ─────────────→ Query\n[1,0,1]                  [1,0,1]\n     │\n     └──── very similar\n\nDocument 1 ─────────────→ Query\n[0,1,1]                  [1,0,1]\n     │\n     └──── partially similar\n\nDocument 2 ─────────────→ Query\n[1,1,0]                  [1,0,1]\n     │\n     └──── partially similar\n\nscores\n[1.00, 0.50, 0.50]         # <<- So Document 0 has cosine similarity 1.0 because it is exactly the same direction as the query.\nshape = (3,)\n'

# Scores: [1.  0.5 0.5] -- 
| Document   | Similarity |
| ---------- | ---------: |
| Document 0 |        1.0 |
| Document 1 |        0.5 |
| Document 2 |        0.5 |

# Top documents: [0 2]   -- Top 2 documents 
Document 0 → 1.0  ← highest
Document 2 → 0.5
Document 1 → 0.5  ← NumPy selected Document 2 for the second position.

# Top scores: [1.  0.5]
Document 0 → Score 1.0
Document 2 → Score 0.5